In [ ]:

# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES
# TO THE CORRECT LOCATION (/kaggle/input) IN YOUR NOTEBOOK,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S R
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

DATA_SOURCE_MAPPING = 'chest-xray-pneumonia:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-data-sets%2F17810%2F23812%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20260311%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20260311T232034Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3D6073015c5b6274ec0f7c77772a83da0c64ca4dbaf2bb08754b14c4d24a32ace2f95e7318e4cfbaaa502a38bb61c705e096e3ce810eeac98579016b6041b6ff42c83e4b64e71b9df39a82bb94e0ad7f658f32252a68646dd455c0e316ff9a4450be19d6d78487f35e71f7ba5e391208bd1b51f1874fffa07a77cbf1c0848983aa3328d8b764cc9622afec251280b4f4cb5adba304f64067f5a14349ccc851bd47b30254d4e73a58b507ccbc57eeb9a0619462131813f8777cd6c195501e954515e2a81ffc7ebddf49ec1042f5cb8e6324f1319d52fcc2c0b92241f0a73ca943e90377f5622fbc5ab45e0c623075693a2486d4bd2d91cec1963ddbb77a44e74e9b'

KAGGLE_INPUT_PATH = '/kaggle/input'
KAGGLE_WORKING_PATH = '/kaggle/working'

system(paste0('sudo umount ', '/kaggle/input'))
system(paste0('sudo rmdir ', '/kaggle/input'))
system(paste0('sudo mkdir -p -- ', KAGGLE_INPUT_PATH), intern=TRUE)
system(paste0('sudo chmod 777 ', KAGGLE_INPUT_PATH), intern=TRUE)
system(
  paste0('sudo ln -sfn ', KAGGLE_INPUT_PATH,' ',file.path('..', 'input')),
  intern=TRUE)

system(paste0('sudo mkdir -p -- ', KAGGLE_WORKING_PATH), intern=TRUE)
system(paste0('sudo chmod 777 ', KAGGLE_WORKING_PATH), intern=TRUE)
system(
  paste0('sudo ln -sfn ', KAGGLE_WORKING_PATH, ' ', file.path('..', 'working')),
  intern=TRUE)

data_source_mappings = strsplit(DATA_SOURCE_MAPPING, ',')[[1]]
for (data_source_mapping in data_source_mappings) {
    path_and_url = strsplit(data_source_mapping, ':')
    directory = path_and_url[[1]][1]
    download_url = URLdecode(path_and_url[[1]][2])
    filename = sub("\\?.+", "", download_url)
    destination_path = file.path(KAGGLE_INPUT_PATH, directory)
    print(paste0('Downloading and uncompressing: ', directory))
    if (endsWith(filename, '.zip')){
      temp = tempfile(fileext = '.zip')
      download.file(download_url, temp)
      unzip(temp, overwrite = TRUE, exdir = destination_path)
      unlink(temp)
    }
    else{
      temp = tempfile(fileext = '.tar')
      download.file(download_url, temp)
      untar(temp, exdir = destination_path)
      unlink(temp)
    }
    print(paste0('Downloaded and uncompressed: ', directory))
}

print(paste0('Data source import complete'))


# **Title: Detecting Paediatric Pneumonia by classification of Linearized Chest X-Ray Images**
---

## **Introduction**

  Pneumonia is a lung disease caused by bacteria that causes an infection in the lungs. The infection causes our air sacs (where oxygen enters into the blood) to fill up with fluid or pus, hence disturbing the process of diffusing oxygen into the blood and making it hard for one to breathe. Pneumonia is the main cause of death for people under the age of 5, with over 800,000 child fatalities due to the disease in 2017 alone. Similarly to cancer, one of the most crucial ways to detect early stages of pneumonia is through chest x-ray images. Through an x-ray scan, a doctor is able to identify inflammation in the lungs and thus able to perform the necessary treatments early. However, inaccurate analysis of the x-ray results may result in an improper diagnosis and decision making, resulting in a costly mistake that could otherwise save lives.

      

  The implementation of clinical-decision support algorithms for medical imaging faces challenges with reliability and interpretability. Here, we establish a diagnostic tool based on a classification framework for the screening of patients’ x-ray to quickly and accurately **identify paedeatric pneumonia from linearized chest x-rays images**.

  We have obtained a database of images of chest x-rays from Kaggle. <https://www.kaggle.com/paultimothymooney/chest-xray-pneumonia>  
    The dataset provided by Kaggle contains about 6000 grey-scaled images, consisting of  `training`, `validating`  and  `testing` sets. Due to the huge amount of data to wrangle (more than 1GB), we are only extracting images from the `training` database, totalling `5216`  labelled images,  to create our own set of training and testing data set that will be used to train our model. There are 3875 `PNEUMONIA` class images and 1341 images of class `NORMAL` specifically.


## **Methods & Results**

### Importing Libraries

We used functions from the  `tidyverse` library to manipulate data frames and tibbles. `repr` is used to resize plots contained in this notebook, such as an  “Accuracy vs K”  graph. The `caret` library ia used to access the train functions to create our classification model. Last but not the least, `imager` allow us to linearize the images into a data frame/ tibble as well as to display the images.



In [ ]:
# import libraries
library(tidyverse)
library(imager)
library(repr)
library(caret)

Data has already been loaded into the server we are using. We start off by using the first image from the NORMAL X-ray results dataset as a test.

In [ ]:
# load normal image
train_NORMAL_dir <- '../input/chest-xray-pneumonia/chest_xray/chest_xray/train/NORMAL/'
train_NORMAL <- list.files(train_NORMAL_dir)

test_image <- load.image(paste(train_NORMAL_dir, train_NORMAL[1], sep=''))
plot(test_image)

<div align="center"> Figure 1: Original class'Normal' X-ray scan </div>

### Resizing images
We proceeded to resize all images from an orignal size of (2090 x 1858) pixels to (20 x 20) pixels as well as cropping the borders, significantly reducing computational time.

In [ ]:
# resizing `normal` image
resize_height <- 20
resize_width <- 20
resized_test_image <- resize(test_image, resize_width, resize_height)
pixels_to_crop <- 0
cropped_test_image <- crop.borders(resized_test_image, nPix = pixels_to_crop) #%>% imsub(x %inr% c(1,3))
new_height <- dim(cropped_test_image)[1]
new_width <- dim(cropped_test_image)[2]
plot(cropped_test_image)

<div align="center"> Figure 2: Resized class 'Normal' X-ray scan </div>

### Creating data frame

Next, we initialize a data frame and fit every picture from the Normal Group into a row in the data frame.
This process is called **linearizing data**. For example, if we split the image into 10 rows and 20 columns, we will have values of the shade of the picture in 10 rows and 20 columns, but then we’ll linearize the data to make it into a single column with 200 columns.

In [ ]:
# create empty data frame of images from class 'normal'
df_NORMAL <- data.frame(matrix(NA, nrow = length(train_NORMAL), ncol = new_height * new_width))

The most important (and lengthy) part of the process of data wrangling is going to be fitting every picture within the working directory into the dataframe we have just created.

In [ ]:
#linearize data
start.time <- Sys.time()
for (i in 1:length(train_NORMAL)) {
    im <- load.image(paste(train_NORMAL_dir, train_NORMAL[i], sep=''))
    thmb <- resize(im, resize_height, resize_width) %>% crop.borders(nPix = pixels_to_crop) #%>% imsub(x %inr% c(1,3))
    df_NORMAL[i,] <- t(as.vector(thmb))
}
end.time <- Sys.time()
interval <- end.time - start.time
print(interval)

We then add a categorical variable, `y`, as an additional column in our data frame to differentiate between the `pneumonia` and `normal` class.  We set the `normal` class to hold a numerical value of `0`.

In [ ]:
# set y = 0 (class = normal)
y <- 0
df_NORMAL <- cbind(df_NORMAL, y)
head(df_NORMAL)

At this point, we have completed linearizing the `normal` class of images. Now, we need to linearize the `pneumonia` class of images. It is pre-processed similarly with the only difference being the `pneumonia` class will have a category of 1 instead of 0. This is done by setting y as 1 instead of 0.

The following code repeats the dataset building process for images of the class `PNEUMONIA`.

In [ ]:
# load images from Kaggle
train_PNEUMONIA_dir <- '../input/chest-xray-pneumonia/chest_xray/chest_xray/train/PNEUMONIA/'
train_PNEUMONIA <- list.files(train_PNEUMONIA_dir)

test_image <- load.image(paste(train_PNEUMONIA_dir, train_PNEUMONIA[1], sep=''))

#resize image
resized_test_image <- resize(test_image, resize_width, resize_height)
cropped_test_image <- crop.borders(resized_test_image, nPix = pixels_to_crop) #%>% imsub(x %inr% c(1,3))

#create data frame for pneumonia images
df_PNEUMONIA <- data.frame(matrix(NA, nrow = length(train_PNEUMONIA), ncol = new_height * new_width))

start.time <- Sys.time()

#linearize data
for (i in 1:length(train_PNEUMONIA)) {
    im <- load.image(paste(train_PNEUMONIA_dir, train_PNEUMONIA[i], sep=''))
    thmb <- resize(im, resize_width, resize_height) %>% crop.borders(nPix = pixels_to_crop) #%>% imsub(x %inr% c(1,3))
    df_PNEUMONIA[i,] <- t(as.vector(thmb))
}
end.time <- Sys.time()
interval <- end.time - start.time
print(interval)

# set y = 1 (class = pneumonia)
y <- 1
df_PNEUMONIA <- cbind(df_PNEUMONIA, y)
head(df_PNEUMONIA)

In [ ]:
plot(test_image)

<div align="center"> Figure 3: Original class 'Pneumonia' X-ray scan </div>

In [ ]:
plot(cropped_test_image)

<div align="center"> Figure 4: Resized class 'Pneumonia' X-ray scan <div align="center">

### Combining Data Frames

So far, we have succeeded in linearizing the normal and pneumonia group images into their respective data frames, one with `X_NORMAL` + `Y_NORMAL`, the other with `X_PNEUMONIA` + `Y_PNEUMONIA`. Since they both underwent the same linearization process, they have the same number of columns (ncols=2) but different number of rows because there are more images for Pneumonia (nrows=3875) than the images for Normal (nrows=1346).

To combine our datasets, we used `rbind` which combines the rows from the normal dataset with the rows from the pneumonia dataset. This process is similar to stacking datasets together.

To make sure that this process is replicable, we use `set.seed`.

In [ ]:
# combine dataset using rbind
set.seed(2020)
dataset <- rbind(df_NORMAL, df_PNEUMONIA) %>%
    mutate(y = as.factor(y))

### Splitting into training and testing sets

As of now, we have a single giant data frame consisting of all the linearized images. If we pass all of the images into our classifier, we would not have any images to use as a measure of how accurate the model is. If we use the data that we have used to train the model before, the model will recognize that image because it has “seen” it before and will most likely predict correctly.

Therefore, to allow a fair process of measuring the model’s accuracy, we must calculate its accuracy based on how accurate it is at predicting images that it hasn’t “seen” before. To do this, we can split the data into `training` and `testing` datasets and only use the training dataset to create our classification model. By doing so, we will have a set of images that the model hasn’t seen before which is the testing dataset. We can then use the testing dataset to estimate how accurate our model is at predicting “new” data. By doing so, we can estimate how accurate the model would be at predicting a new set of data.

To do this, we will use the golden ratio which is a 75-25% ratio of training and testing datasets respectively. We will use the function `createDataPartititon( )` to split our dataset by obtaining the rows that are considered training. We will name these rows as `training_rows`. We will then use the `slice( )` function to extract those rows specified by the training_rows. We use `slice(training_rows)` to get the training data set and use `slice(-training_rows)` to get the testing data set.

In [ ]:
# split data into training and testing sets
set.seed(2020)
training_data <- dataset %>%
    select(y) %>%
    unlist() %>%
    createDataPartition(p = 0.75, list = FALSE)
X_train <- dataset %>%
    select(-y) %>%
    slice(training_data) %>%
    data.frame()
Y_train <- dataset %>%
    select(y) %>%
    slice(training_data) %>%
    unlist()
X_test <- dataset %>%
    select(-y) %>%
    slice(-training_data) %>%
    data.frame()
Y_test <- dataset %>%
    select(y) %>%
    slice(-training_data) %>%
    unlist()


### Scaling the dataset

Scaling data is an important part of creating classification models. The knn classification determines its nearest neighbors based on the Euclidean distance between the point and its neighbors and chooses the least distance. This means that if for example, we are measuring the height and length of an apple (similar orders of magnitude) but instead, we measure height in centimeters and measure length in meters. We would then get that the height is 10 and the length is 0.01. A change in length will then be of smaller importance than a change in height because the height is bigger in scale. To avoid this, we need to scale our data.

In our case, our “shade scale” to determine how dark our images are was the same one used throughout to linearize the data but just incase it might affect the data, we will still linearize it. To scale our dataset, we will need to use the preProcess function to specify how the data should be manipulated in order to scale it properly. Then, we will use the predict() function to scale our X_training set and the X_testing set.

The preProcess function will know how to scale it by seeing the training set and determining how to scale it. We pass the training set instead of the testing set into preProcess because we want to ensure that our classification model has never seen the testing dataset AT ALL and by making the scaling process based on the training dataset, we can ensure that the model still hasn’t seen the testing dataset.



### Balancing the dataset

Balancing data also affects our classification model if one data is heavily over dominated as compared to the other categories. For example, one category only has 1 data point and another has 50 data points. Hence, if we choose a k value of 3, no matter what point, the one category data point will always be outnumbered and hence, no data passed to the model will ever be classified in the outnumbered category. This means that the model is unbalanced.

In our case, the normal group is outnumbered about 1:3 as compared to the pneumonia group. Hence, balancing our data will allow for more balanced results.

### Knn classification

Firstly, we’ll need to make a data frame for the range of values of k that we would like to try to determine the optimum k for our model. We have decided on the range of k from 1 to 21. We will be using `seq` function to create our dataframe.

In [ ]:
set.seed(2020)
k <- data.frame(k = seq(from = 1, to = 21, by = 1))

We used `trainControl` to assess the accuracy of the model with different values of k. Our train control will be using a 5-fold cross validation in order to obtain better results. The method `cv` corresponds to cross validation.

In [ ]:
set.seed(2020)
train_control <- trainControl(method = "cv", number = 5)

We will then pass all of these arguments into the `train( )` function to create our model. The `method = “knn”` corresponds to the fact that we are doing knn classification. After we create our model, we can then extract the accuracy column of our model to see how accurate our model was for different values of k.

In [ ]:
set.seed(2020)
selecting_k <- train(x = X_train, y = Y_train, method = "knn", tuneGrid = k, trControl = train_control)
k_accuracies <- selecting_k$results %>%
                     select(k, Accuracy)

### Visualize the data

We constructed an “Accuracy vs K” plot to decide which value of k will be the optimal for our model. We chose the highest point in the graph as the optimum value of K as the highest point means it has the highest accuracy. We will be using `ggplot` to create out plot. We will be creating a line and point plot because the line will help us determine the highest point more easily and the points will help us determine the corresponding value of K for that peak point. The arguments for ggplot will be `geom_point( )` for the scatter plot, and `geom_line( )` for the line plot.

In [ ]:
set.seed(2020)
accuracy_k_plot <- k_accuracies %>% ggplot(aes(x = k,y = Accuracy)) + geom_point() +
                   geom_line() + labs(x = "Value of K", y = "Accuracy") +
                    ggtitle('Plot of Accuracy versus K')
accuracy_k_plot

<div align="center"> Figure 5: Plot of Accuracy versus K </div>

As observed from the graph above, the highest point on the plot is at k=”10”. Hence we will be using this as the optimum value of k to create our final classification model.

To create our final classification model, we will need to make another dataframe consisiting of only the chosen value of k.

In [ ]:
set.seed(2020)
chosen_k <- data.frame(k = 10)

We need to use the train( ) function again to create our classification model. This time, we don’t need to have the trainControl argument inside the train function because we have already selected a single value of k as our optimum value.


In [ ]:
set.seed(2020)
knn_model <- train(x = X_train, y = Y_train, method = "knn", tuneGrid = chosen_k)

Finally, we will pass the testing dataset into our final classification model. The model will predict the categories they should belong using the predict() function and using the confusionMatrix( ) function, we will compare its predictions with the actual labels.

In [ ]:
# final classification  model
set.seed(2020)
Y_predicted <- predict(knn_model, X_test)
results <- confusionMatrix(data = Y_predicted, reference = Y_test)
results

Based on the confusion matrix above, the accuracy of our model obtained is 92.48%. From this accuracy value alone, we can infer that the model seems quite robust. However the sensitivity of our model is at 76.72%. From this we can infer that our model has more to improved on.

### **Analysis**

#### Correct and wrong predictions

To analyze the results of our model further, we want to see how many of the data in the testing dataset was predicted correctly by the model and how many were predicted wrong. We then want to see out of the ones that are predicted correctly, how many of them came from the one category and how many of them came from the other category. Similarly, we will do this for the ones predicted wrong. We will then be able to detect any possible imbalances.


In [ ]:
set.seed(2020)
labels <- Y_test %>% data.frame()
predictions <- Y_predicted %>% data.frame()

# data frame of correc
correct <- which(predictions == labels) %>% data.frame()
count <- correct %>% summarise(n = n())

#list of wrongly labelled images
wrong <- which(predictions != labels) %>% data.frame()
wrong_count <- wrong %>% summarise(n = n())

count
wrong_count


Hurray! We have completed the classification of our testing set, with a total of 1205 images predicted with the correct class labels, and 98 images predicted wrongly, achieving an accuracy of 92.48%.

We also would like to know what was the factor behind the images that were predicted correctly and the images that were wrongly predicted. The best way to do this would be to print out the images that were predicted correctly and the ones predicted wrongly and then compare them directly.

In [ ]:
# list of wrongly predicted images
head(wrong)

As an example, we analyzed the 7th image from our list of wrongly predicted images.

In [ ]:
look.for <- bind_cols(X_test, Y_test %>% data.frame()) %>% slice(7)
look.for

The last column which determines the class label has a value of 0. Therefore it is under class `Normal`. The model has made an error and  incorrectly predicted the image `pneumonia`.

We can further analyze this example with an actual x-ray image which has class `pneumonia`.



In [ ]:
library(prodlim)
row.match(look.for, df_NORMAL, nomatch = 0)


In [ ]:
# obtain image of class PNEUMONIA
compare_wrong_image <- load.image(paste(train_NORMAL_dir, train_NORMAL[23], sep=''))


Now lets get an image of a correctly predicted pneumonia X-ray to make a comparision

In [ ]:
# correctly classified x-ray images
head(correct)

Unforutanely all these values in the first 6 slices were of class`NORMAL`. Therefore the entire list had to be printed out to obtain one `PNEUMONIA` class example. Lucky slice 1298 has been chosen!

In [ ]:
look.for <- bind_cols(X_test, Y_test %>% data.frame()) %>% slice(1298)
look.for

In [ ]:
row.match(look.for, df_PNEUMONIA, nomatch = 0)

In [ ]:
compare_correct_image <- load.image(paste(train_PNEUMONIA_dir, train_PNEUMONIA[3830], sep=''))

In [ ]:
plot(compare_correct_image)

<div align="center"> Figure 6: Correctly predicted pneumonia x-ray </div>

In [ ]:
plot(compare_wrong_image)

<div align="center"> Figure 7: Inorrectly predicted NORMAL x-ray </div>

We do see some similarity and definitely this would've been difficult to be detected by the human eye as well. Now let's try working on an expanded grayscale view.

In [ ]:
df <- grayscale(compare_correct_image) %>% as.data.frame
p <- ggplot(df,aes(x,y))+geom_raster(aes(fill=value))
p <- p+scale_x_continuous(expand=c(0,0))+scale_y_continuous(expand=c(0,0),trans=scales::reverse_trans())
p+scale_fill_gradient(low="black",high="white")

In [ ]:
df <- grayscale(compare_wrong_image) %>% as.data.frame
p <- ggplot(df,aes(x,y))+geom_raster(aes(fill=value))
p <- p+scale_x_continuous(expand=c(0,0))+scale_y_continuous(expand=c(0,0),trans=scales::reverse_trans())
p+scale_fill_gradient(low="black",high="white")


In [ ]:
px1 <- (isoblur(compare_wrong_image,4)  > .5 )
plot(px1)


In [ ]:
px2 <- (isoblur(compare_correct_image,4)  > .5 )
plot(px2)

[TODO] We can now clearly see where the predictor has mispredicted. This type of error cannot be avoided unless there is a pixel to pixel image processing algorithm. We now have used the `isoblur` fucntion to highlight the pixels. This clearly suggests that overall our prediction model has done a decent job for image classification using K-NN.


## Discussion
-  Summary of Results
    
Overall, we were satisfied and quite suprised that our simple knn-classification mode was able to achieve an accuracy model of 92.48%, successfully predict a total of 1205 images with the correct class labels, and 98 images predicted wrongly, thus were able to detect accurately paediatric pneumonia.

As for the wrong classification, we were able to determine the cause of error through an expanded greyscale view and make an inference that the magnitude of pixelation can influence the classfication model. This may be a result of our attempts to resize tha image in order to reduce processing time.

Our model does have its shortcomings in that our training datasets consists of [TODO] `PNEUMONIA` class images and .. `NORMAL` class images. There might be a case of overfitting as we can see from our plot [TODO] ?  

Computationally the image linearization process took the longest time (8 mins) but was not a big setback in our classification process. In the future, it is possible to focus our image processing on a cropped part of the model (part that encompasses the whole boundary of the lungs), as we may be wasting valuable computation time by processing the whole x-ray image.

- Impact of Results
Computer aided diagnosis a huge leap in the medical industry as well as any field requiring the techniques of image identification. Our classification model can be test on other pulmonary based diseases such as chronic bronchitis, bronchiectasis, congestive heart failure and more.

KNN classification is very advantages in that it is resilient to noisy and effective data training when the training data is large, which is very common in the biomedical field as there is huge amount of data and samples available. However, the value of the parameter K (number of nearest neighbors) has to be determined to execute the KNN classifier. Furthermore, on an image classification aspect, no specific variables could be added on top of the linearized images such as history of health diseases and gene inheritance disorders.that could make the classification more accurate.


- Further Projects based on Results

Since we only processed a subset of the data due to server and computational power limitations, our next step would hopefully aim towards increase our computational efficiency either through a hardware upgrade or through BigQuery to process larger datasets.

We have been looking at diving into deep learning on image processing as there have been major advancement in image classification using convolution neautral networks(CNN) such as Keras and Tensorflow. It would be interesting to compare the accuracy of other types of classification with that of knn classification and adopt methodologies based on what works best.




## References
Rudan I. Boschi-Pinto C. Biloglav Z. Mulholland K. Campbell H. Epidemiology and etiology of childhood pneumonia. Bull. World Health Organ. 2008; 86: 408-416

Rajpurkar P, Irvin J, Zhu K, Yang B, Mehta H, Duan T, et al. CheXNet: Radiologist-Level Pneumonia Detection on Chest X-Rays with Deep Learning; 2017 [cited 1 July 2018]. Preprint. Available from: https://arxiv.org/abs/1711.05225.